# Five-color WCA lattice-ABP MIPS demo (Google Colab)

Colab runtime type must be set to **GPU**. This version uses the Torch CUDA tensor path (`backend="torch"`); it does not compile or use the legacy random-sequential CUDA extension.

## Repository setup

Place the current `CNEEP_v2` directory in Google Drive at `MyDrive/CNEEP_v2` before running this notebook. The cell mounts Drive, installs only Python dependencies, and imports the repository directly; no `nvcc`, compiler, or CUDA extension build is required.

In [ ]:
from pathlib import Path
import sys

from google.colab import drive
drive.mount('/content/drive')

ROOT = Path('/content/drive/MyDrive/CNEEP_v2')  # change only if stored elsewhere
if not (ROOT / 'data/lattice_abp_tc/core.py').is_file():
    raise RuntimeError(f'Cannot find CNEEP_v2 at {ROOT}. Copy the repository to that Drive path or edit ROOT.')
%pip -q install -r {ROOT / 'data/lattice_abp_tc/requirements-l40s.txt'}
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


In [ ]:
import json
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch

from data.lattice_abp_tc import ThermodynamicLatticeABP, ThermodynamicLatticeABPParams

if not torch.cuda.is_available():
    raise RuntimeError('No GPU is visible. In Colab choose Runtime > Change runtime type > GPU, then reconnect.')
DEVICE = torch.device('cuda:0')
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda, '| GPU:', torch.cuda.get_device_name(DEVICE))


In [ ]:
def n_from_packing_fraction(phi, box_size, sigma):
    return int(round(phi * 4.0 * box_size * box_size / (math.pi * sigma * sigma)))

L, grid_size, sigma, phi = 16.0, 32, 0.5, 0.30
params = ThermodynamicLatticeABPParams(
    N=n_from_packing_fraction(phi, L, sigma), L=L, grid_size=grid_size, sigma=sigma,
    epsilon=1.0, v0=50.0, Dr=1.5, Dt=1.0, dt=1.0e-4, prefactor='cv',
    seed=7, device=str(DEVICE), dtype='float32', backend='torch',
)

quick_run = True  # Set False only after the short run succeeds.
n_steps = 200 if quick_run else 200_000
burn_in = 0 if quick_run else 100_000
save_interval = 10 if quick_run else 100
B = 1
sim = ThermodynamicLatticeABP(params)
print(f'device={sim.device}, backend={sim.backend}, N={params.N}, phi={params.phi:.3f}, Pe={params.Pe:.2f}')


In [ ]:
torch.cuda.synchronize(DEVICE)
started = time.perf_counter()
result = sim.simulate(
    B=B, burn_in=burn_in, n_steps=n_steps, save_interval=save_interval,
    show_progress=True, save_diagnostics=False, save_occupancy=True,
    save_exact_medium_ep=True,
)
torch.cuda.synchronize(DEVICE)
elapsed = time.perf_counter() - started
print(f'elapsed: {elapsed:.2f} s | {B * n_steps / elapsed:.1f} ensemble steps/s')


In [ ]:
summary = {
    'initial': sim.mips_summary_from_sites(result['sites'][0], include_coarse=False),
    'final': sim.mips_summary_from_sites(result['sites'][-1], include_coarse=False),
    'mean_exact_medium_ep_rate': result['exact_medium_ep_rate'].mean(dim=1).cpu().numpy().tolist(),
}
print(json.dumps(summary, indent=2))

fig, axes = plt.subplots(1, 2, figsize=(10, 5), constrained_layout=True)
for ax, occ, title in zip(axes, (result['occupancy'][0, 0], result['occupancy'][-1, 0]), ('initial', 'final')):
    ax.imshow(occ.T.cpu().numpy(), origin='lower', cmap='gray_r', vmin=0, vmax=1, interpolation='nearest')
    ax.set_title(title + ' occupancy')
    ax.set_axis_off()
plt.show()


In [ ]:
OUTDIR = ROOT / 'output' / 'lattice_abp_tc_colab'
OUTDIR.mkdir(parents=True, exist_ok=True)
np.savez_compressed(OUTDIR / 'trajectory.npz',
    sites=result['sites'].cpu().numpy(), occupancy=result['occupancy'].cpu().numpy(),
    theta=result['theta'].cpu().numpy(), times=result['times'].cpu().numpy(),
    exact_medium_ep=result['exact_medium_ep'].cpu().numpy(),
    exact_medium_ep_rate=result['exact_medium_ep_rate'].cpu().numpy())
(OUTDIR / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('saved:', OUTDIR)
